## Представление реальных данных с помощью тензоров
- Представление реальных данных в виде тензоров PyTorch
- Работа с различными типами данных
- Загрузка данных из файла
- Преобразование данных в тензоры
- Приведение тензоров к форме, пригодной для использования в качестве входных данных для моделей нейронных сетей

### Глава 1: Работа с изображениями

Изображение — это, по сути, сетка чисел, расположенных в строках и столбцах и представляющих пиксели. Каждый пиксель может содержать либо одно скалярное значение (что формирует полутоновое изображение), либо несколько скалярных значений (которые обычно соответствуют различным цветам или другим характеристикам, например, данным о глубине, получаемым со специализированных камер).

In [14]:
import torch
# загрука файла с изображением
# C:\Users\excavator\projects\cv\03_dl_pytourch\.venv\Scripts\python.exe -m pip install imageio

import imageio.v2 as imageio

img_arr = imageio.imread('./data/p1ch4/image-dog/bobby.jpg')
# img_arr здесь уже numpy array
img_arr.shape

(720, 1280, 3)

Модули PyTorch, предназначенные для работы с изображениями, требуют, чтобы тензоры имели структуру C × H × W: каналы, высота и ширина

In [15]:
# метод permute тензора для правильной компоновки сначала подключите канал 2, а затем каналы 0 и 1:

img = torch.from_numpy(img_arr)
out = img.permute(2, 0, 1) # нет компированя, только stride
out.shape

torch.Size([3, 720, 1280])

Чтобы создать набор данных, состоящий из множества изображений, создается тензор размерности N × C × H × W.

In [16]:
# предварително готовим пачку заполненую нулями
import os
data_dir = './data/p1ch4/image-cats/'
png_files = [f for f in os.listdir(data_dir) if f.endswith('.png')]
batch = torch.zeros(len(png_files), 3, 256, 256, dtype=torch.uint8)

In [17]:
# считываем данные из всех файлов
for i, filename in enumerate(png_files):
    img_arr = imageio.imread(os.path.join(data_dir, filename))
    img_t = torch.from_numpy(img_arr)
    img_t = img_t.permute(2, 0, 1)
    # удаляем все лишнее
    img_t = img_t[:3]
    batch[i] = img_t

Нормализация данных

Нейронные сети демонстрируют наилучшие результаты при обучении, если входные данные находятся в диапазоне примерно от 0 до 1 или от –1 до 1. 
Сложность нормализации зависит от того, какой диапазон применяется -от 0 до 1 или от –1 до 1. 
Один из вариантов — разделить значения пикселей на 255 (максимальное число, представимое в 8-битном формате без знака):

In [18]:
batch = batch.float()
batch /= 255.0

Ещё один вариант — вычислить среднее значение и стандартное отклонение входных данных и масштабировать их так, чтобы для каждого канала среднее значение стало равным нулю, а стандартное отклонение — единице; этот метод обычно называют стандартизацией:

In [19]:
n_channels = batch.shape[1]
for c in range(n_channels):
    mean = torch.mean(batch[:, c])
    std = torch.std(batch[:, c])
    batch[:, c] = (batch[:, c] - mean) / std

С входными данными можно выполнять и другие операции — например, геометрические преобразования: повороты, масштабирование, кадрирование. Они могут быть полезны при обучении или необходимы для того, чтобы привести произвольные входные данные в соответствие с требованиями нейросети (например, подогнать размер изображения).

### Глава 2: 3D-изображения: объемные данные
стр 77

3D - серия изображений, расположенные вдоль оси.
Появляется лишь дополнительное измерение — глубина (depth), следующее за размерностью каналов в 5D-тензоре размерности N × C × D × H × W.

Загрузка данных в специализированном формате
Например, образец данных компьютерной томографии (КТ). В качестве аргумента эта функция принимает путь к директории, собирает все файлы в формате DICOM и объединяет их в трехмерный массив NumPy.

In [22]:
import imageio
dir_path = "./data/p1ch4/volumetric-dicom/2-LUNG 3.0  B70f-04083"
vol_arr = imageio.volread(dir_path, 'DICOM')
vol_arr.shape

Reading DICOM (examining files): 1/99 files (1.0%13/99 files (13.1%26/99 files (26.3%39/99 files (39.4%52/99 files (52.5%64/99 files (64.6%77/99 files (77.8%89/99 files (89.9%99/99 files (100.0%)
  Found 1 correct series.
Reading DICOM (loading data): 77/99  (77.899/99  (100.0%)


(99, 512, 512)

In [ ]:
# PyTorch ожидает наличия информации о каналах, а в текущих данных она отсутствует.
# unsqueeze добавляет новое измерение размера 1 в указанную позицию.
# операция преобразует трехмерный массив с формой [D, H, W] в четырехмерный тензор с формой [1, D, H, W], где первое измерение соответствует каналу.
vol = torch.from_numpy(vol_arr).float()
vol = torch.unsqueeze(vol, 0)

vol.shape

torch.Size([1, 99, 512, 512])

### Глава 3: Представление табличных данных

Для примера исползуется набор данных «Качество вина» (Wine Quality).
Файл представляет собой набор значений, 12 столбцов. Первые 11 столбцов содержат значения химических показателей, а последний столбец — оценку качества по шкале от 0 (очень плохо) до 10 (отлично).

In [ ]:
import csv
import numpy as np
wine_path = "./data/p1ch4/tabular-wine/winequality-white.csv"
# Ззадаем тип данных, разделитель, а также указываем, что первую строку считывать не нужно.
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";",
skiprows=1)
wineq_numpy

array([[ 7.  ,  0.27,  0.36, ...,  0.45,  8.8 ,  6.  ],
       [ 6.3 ,  0.3 ,  0.34, ...,  0.49,  9.5 ,  6.  ],
       [ 8.1 ,  0.28,  0.4 , ...,  0.44, 10.1 ,  6.  ],
       ...,
       [ 6.5 ,  0.24,  0.19, ...,  0.46,  9.4 ,  6.  ],
       [ 5.5 ,  0.29,  0.3 , ...,  0.38, 12.8 ,  7.  ],
       [ 6.  ,  0.21,  0.38, ...,  0.32, 11.8 ,  6.  ]],
      shape=(4898, 12), dtype=float32)

In [26]:
# все ли данные были считаны:
col_list = next(csv.reader(open(wine_path), delimiter=';'))

wineq_numpy.shape, col_list

((4898, 12),
 ['fixed acidity',
  'volatile acidity',
  'citric acid',
  'residual sugar',
  'chlorides',
  'free sulfur dioxide',
  'total sulfur dioxide',
  'density',
  'pH',
  'sulphates',
  'alcohol',
  'quality'])

In [27]:
# массив NumPy в тензор PyTorch:
wineq = torch.from_numpy(wineq_numpy)

wineq.shape, wineq.dtype

(torch.Size([4898, 12]), torch.float32)

Непрерывные, порядковые и категориальные величины

При анализе данных важно различать три типа числовых величин. Первый тип — непрерывные величины. Они наиболее интуитивно понятны, когда представлены в виде чисел. Для них характерен строгий порядок, а разность между значениями имеет точный математический смысл. Например, при измерении температуры разница между 20°C и 25°C соответствует такому же физическому изменению тепловой энергии, как и разница между 30°C и 35°C. Аналогичным образом, при измерении роста разница в 5 см имеет одинаковый физический смысл, независимо от того, сравниваем ли мы рост 150 см и 155 см или 185 см и 190 см.
В научной литературе непрерывные величины подразделяются на более узкие категории. Если имеет смысл утверждение, что одна величина в два раза больше другой — как в случае с весом (10 кг в два раза тяжелее 5 кг) или расстоянием (30 миль в три раза больше, чем 10 миль), — то говорят, что такие величины представлены в шкале отношений. Температура по шкале Цельсия, напротив, обладает свойством постоянной разности, однако нельзя утверждать, что 20°C — это «в два раза жарче», чем 10°C (поскольку 0°C не означает полное отсутствие тепла); поэтому шкала Цельсия является лишь интервальной шкалой.

Далее следуют порядковые величины. В них сохраняется строгий порядок, свойственный непрерывным величинам, но фиксированная математическая взаимосвязь между разностями значений уже отсутствует. Хороший пример — выбор размера напитка: «маленький», «средний» или «большой», где «маленькому» соответствует значение 1, «среднему» — 2, а «большому» — 3. «Большой» напиток действительно больше «среднего» (подобно тому, как число 3 больше числа 2), однако это не дает нам никакой информации о том, насколько именно он больше. Если бы мы заменили наши значения «1», «2» и «3» на реальные объемы (скажем, 8, 12 и 24 жидкие унции), они превратились бы в интервальные величины. Важно помнить, что с такими значениями нельзя выполнять математические операции — допустимо лишь их упорядочивание. Так, если «большой» напиток обозначить как 3, а «маленький» — как 1, то их среднее значение вовсе не будет соответствовать напитку «среднего» размера!

Наконец, существуют категориальные признаки, значения которых не имеют ни порядка, ни числового смысла. Зачастую это просто перечисление возможных вариантов, которым произвольно присвоены определенные числа. Примером может служить такое кодирование: вода — 1, кофе — 2, газировка — 3, молоко — 4. В том, чтобы поставить воду на первое место, а молоко — на последнее, нет никакой глубокой логики; нам просто нужны различные значения, чтобы различать эти категории. Мы могли бы присвоить кофе значение 10, а молоку — -3, и по сути ничего бы не изменилось (хотя использование значений в диапазоне от 0 до N–1 дает определенные преимущества при применении метода one-hot encoding или создании эмбеддингов, о чем мы поговорим в разделе 4.5.4). Поскольку числовые значения здесь не несут содержательного смысла, говорят, что они представлены в номинальной шкале.

Представление оценок

Оценку можно рассматривать как непрерывную переменную (вещественное число) и решать задачу регрессии, либо же воспринимать её как метку и пытаться определить эту метку на основе химического анализа в рамках задачи классификации. 
В обоих случаях оценку, как правило, исключают из тензора входных данных и сохраняют в отдельном тензоре — таким образом, она служит эталонным значением (ground truth) и не подаётся на вход модели.

In [28]:
# Выбирает все строки и все столбцы, кроме последнего
data = wineq[:, :-1]
data, data.shape

(tensor([[ 7.0000,  0.2700,  0.3600,  ...,  3.0000,  0.4500,  8.8000],
         [ 6.3000,  0.3000,  0.3400,  ...,  3.3000,  0.4900,  9.5000],
         [ 8.1000,  0.2800,  0.4000,  ...,  3.2600,  0.4400, 10.1000],
         ...,
         [ 6.5000,  0.2400,  0.1900,  ...,  2.9900,  0.4600,  9.4000],
         [ 5.5000,  0.2900,  0.3000,  ...,  3.3400,  0.3800, 12.8000],
         [ 6.0000,  0.2100,  0.3800,  ...,  3.2600,  0.3200, 11.8000]]),
 torch.Size([4898, 11]))

In [29]:
# Выбирает все строки и последний столбец
target = wineq[:, -1]
target, target.shape

(tensor([6., 6., 6.,  ..., 6., 7., 6.]), torch.Size([4898]))

In [30]:
# Преобразовать тензор в тензор меток можно путем предоставления метки в виде целочисленного вектора значений:
target = wineq[:, -1].long()
target

tensor([6, 6, 6,  ..., 6, 7, 6])

Другой подход заключается в использовании прямого кодирования (one-hot encoding) оценок: 
каждая из 10 возможных оценок представляется в виде вектора из 10 элементов, где все элементы равны 0, за исключением одного, причем для каждой оценки этот ненулевой элемент находится на своей уникальной позиции (индексе). 
Так, оценке 1 может соответствовать вектор (1,0,0,0,0,0,0,0,0,0), оценке 5 — вектор (0,0,0,0,1,0,0,0,0,0) и так далее.

In [32]:
target.shape[0]

4898

In [31]:
# Метод scatter_ заполняет тензор значениями из исходного тензора в соответствии с индексами, переданными в качестве аргументов:
target_onehot = torch.zeros(target.shape[0], 10)
target_onehot

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [ ]:
target_onehot.scatter_(1, target.unsqueeze(1), 1.0)
random_indices = torch.randint(0, len(target), (5,))
# 5 случайных оценок качества вина :
print(target[random_indices])
# Первые 5 оценок качества вина (в формате one-hot encoding):
print(target_onehot[random_indices])

tensor([6, 6, 6, 7, 7])
tensor([[0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])


К первому элементу тензора `target` мы обращаемся как `target[0]`, а к первому элементу того же тензора после применения `unsqueeze` — как `target_unsqueezed[0,0]`.

In [35]:
target_unsqueezed = target.unsqueeze(1)
target_unsqueezed

tensor([[6],
        [6],
        [6],
        ...,
        [6],
        [7],
        [6]])

Когда выполнять категоризацию
стр 85